# 26 - Massey Products, Sullivan Minimal Models & Rational Homotopy

Rational homotopy theory (Sullivan, Quillen) answers the question: *how much of a space's homotopy theory is determined by algebra?* The answer is: *everything, over the rationals*. A simply-connected space $X$ is completely determined up to rational equivalence by its **Sullivan minimal model** — a free commutative DGA $(\wedge V, d)$ that encodes all rational homotopy groups and all Massey products.

This notebook covers `pysurgery.core.rational_homotopy` in full: DGA constructions, Sullivan models, formality, Massey products, and rational homotopy groups.

## Learning Goals
- Understand DGAs and Sullivan minimal models as algebraic models for spaces
- Build cohomology algebras with `sphere_cohomology`, `complex_projective_space_cohomology`
- Compute Sullivan minimal models with `sullivan_minimal_model`
- Certify formality with `is_formal_space` → `FormalityResult`
- Extract Massey products with `extract_massey_products` → `MasseyProductsResult`
- Compute rational homotopy groups $\pi_n(X) \otimes \mathbb{Q}$ with `rational_homotopy_group`

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Simply-connected space $X$ | `SimplicialComplex` |
| **Algebraic** | Sullivan minimal model $(\wedge V, d)$ | `sullivan_minimal_model` |
| **Result** | Formality certificate + Massey product list | `is_formal_space`, `extract_massey_products` |

## Formal Background

A **Sullivan minimal model** for $X$ is a free CDGA $(\wedge V, d)$ with:
- $V = \bigoplus_{n \geq 1} V^n$ (graded vector space, finitely many generators per degree)
- $d(V^n) \subset \wedge^{\geq 2} V^{<n}$ (minimality condition)
- $H^*(\wedge V, d) \cong H^*(X; \mathbb{Q})$ (quasi-isomorphism)

The indecomposable part of $\wedge V$ is dual to $\pi_*(X) \otimes \mathbb{Q}$.

In [ ]:
from pysurgery.core.rational_homotopy import (
    RationalMinimalModelResult, sullivan_minimal_model,
    sphere_cohomology, complex_projective_space_cohomology, product_cohomology,
    is_formal_space, FormalityResult,
    extract_massey_products, MasseyProductsResult, rational_homotopy_group, RationalHomotopyGroup,
)

print("=" * 70)
print("26 - Massey Products, Sullivan Models & Rational Homotopy: Ready")
print("=" * 70)

## Part 1: Cohomology Algebras

Before building the Sullivan model, you need a cohomology algebra. The module provides standard cohomology algebras as `RationalCohomologyAlgebra` objects with generators, relations, and degree data.

In [ ]:
# Standard spaces
H_S3 = sphere_cohomology(n=3)
H_CP2 = complex_projective_space_cohomology(n=2)
H_S2xS3 = product_cohomology(sphere_cohomology(2), sphere_cohomology(3))

print("H*(S³; Q):")
print(f"  generators: {[(g.name, g.degree) for g in H_S3.generators]}")
print(f"  Betti numbers: {H_S3.betti_numbers}")
print()
print("H*(CP²; Q):")
print(f"  generators: {[(g.name, g.degree) for g in H_CP2.generators]}")
print(f"  relations: {H_CP2.relations}")
print(f"  Betti numbers: {H_CP2.betti_numbers}")
print()
print("H*(S²×S³; Q):")
print(f"  Betti numbers: {H_S2xS3.betti_numbers}")

## Part 2: Sullivan Minimal Models

`sullivan_minimal_model` builds the minimal model iteratively: generators are added degree by degree to kill cohomology classes, and differentials are chosen to enforce $d^2 = 0$ and quasi-isomorphism.

In [ ]:
# Sullivan model of CP²: should have generators x (degree 2) and y (degree 5)
# with d(x) = 0 and d(y) = x³
model_CP2: RationalMinimalModelResult = sullivan_minimal_model(H_CP2, max_degree=8)

print("Sullivan minimal model of CP²:")
print(f"  generators: {[(g.name, g.degree) for g in model_CP2.generators]}")
for gen in model_CP2.generators:
    diff = model_CP2.differential_map.get(gen.name, "0")
    print(f"  d({gen.name}) = {diff}")
print(f"  exact: {model_CP2.exact}")
print(f"  theorem_tag: {model_CP2.theorem_tag}")
print()

# Model of S³: generator x₃ (degree 3) with d(x₃)=0
# plus generator y₆ (degree 6) with d(y₆)=x₃²
model_S3: RationalMinimalModelResult = sullivan_minimal_model(H_S3, max_degree=10)
print("Sullivan minimal model of S³:")
for gen in model_S3.generators:
    diff = model_S3.differential_map.get(gen.name, "0")
    print(f"  d({gen.name}) = {diff}  (degree {gen.degree})")

## Part 3: Formality

A space is **formal** if its rational homotopy type is determined by its cohomology ring — i.e., $(\wedge V, d) \simeq (H^*(X), 0)$ as CDGAs. Kähler manifolds (and hence $CP^n$, Riemann surfaces) are formal. Non-formal spaces have non-trivial Massey products that detect the "complexity" of the differential.

`is_formal_space` certifies formality or returns a `FormalityResult` with the obstruction Massey product.

In [ ]:
# Formal spaces
formal_tests = {
    "S²":    sphere_cohomology(2),
    "S³":    sphere_cohomology(3),
    "CP²":   complex_projective_space_cohomology(2),
    "S²×S³": product_cohomology(sphere_cohomology(2), sphere_cohomology(3)),
}

print(f"{'Space':<12} {'is_formal':<12} {'obstruction'}")
print("-" * 50)
for name, H in formal_tests.items():
    f: FormalityResult = is_formal_space(H)
    obs = str(f.obstruction)[:40] if f.obstruction else "None"
    print(f"{name:<12} {str(f.is_formal):<12} {obs}")

print()
print("All standard Kähler manifolds are formal (Deligne-Griffiths-Morgan-Sullivan 1975).")

## Part 4: Massey Products

For a Sullivan model $(\wedge V, d)$, a **triple Massey product** $\langle [a], [b], [c] \rangle$ exists when $d(x) = a \cdot b$ and $d(y) = b \cdot c$ for some elements $x, y$ — then $a \cdot y + x \cdot c$ is a cocycle and its class is the Massey product (up to indeterminacy).

`extract_massey_products` reads decomposable differentials of the minimal model to enumerate all such products systematically.

In [ ]:
# Extract Massey products from the Sullivan model of S³
massey_S3: MasseyProductsResult = extract_massey_products(model_S3)
print(f"Massey products for S³: {len(massey_S3.entries)} entries")
for entry in massey_S3.entries:
    print(f"  ⟨{entry.input_classes}⟩: order={entry.order}, degree={entry.degree}")
print(f"  exact: {massey_S3.exact}")
print()

# Massey products for CP²
massey_CP2: MasseyProductsResult = extract_massey_products(model_CP2)
print(f"Massey products for CP²: {len(massey_CP2.entries)} entries (expected 0 — formal)")
print(f"  exact: {massey_CP2.exact}")
print()

# Summary: which spaces have non-trivial Massey products?
print("Massey product count by space:")
for name, H in formal_tests.items():
    model = sullivan_minimal_model(H, max_degree=8)
    m = extract_massey_products(model)
    print(f"  {name:<12}: {len(m.entries)} products")

## Part 5: Rational Homotopy Groups

The indecomposable elements of $V$ in the Sullivan model $(\wedge V, d)$ are dual to $\pi_*(X) \otimes \mathbb{Q}$. This gives a direct computation of rational homotopy groups.

`rational_homotopy_group(H, n)` returns a `RationalHomotopyGroup` with rank (the rank of $\pi_n(X) \otimes \mathbb{Q}$) and exact flag.

In [ ]:
print("Rational homotopy groups π_n(X) ⊗ Q:")
print()
spaces_and_homotopy = {
    "S²":  (sphere_cohomology(2),  range(2, 8)),
    "S³":  (sphere_cohomology(3),  range(3, 9)),
    "CP²": (complex_projective_space_cohomology(2), range(2, 8)),
}

for name, (H, degrees) in spaces_and_homotopy.items():
    print(f"{name}:")
    for n in degrees:
        pi_n: RationalHomotopyGroup = rational_homotopy_group(H, n=n)
        if pi_n.rank > 0:
            print(f"  π_{n} ⊗ Q = Q^{pi_n.rank}  (exact={pi_n.exact})")
    print()

print("S²: π₂⊗Q=Q, π₃⊗Q=Q (Hopf), all others 0")
print("S³: π₃⊗Q=Q, all others 0 (odd sphere, rationally K(Z,3))")
print("CP²: π₂⊗Q=Q (generator), π₅⊗Q=Q (Whitehead product), others 0")

## Summary Checklist

- [x] Built cohomology algebras with `sphere_cohomology`, `complex_projective_space_cohomology`
- [x] Computed Sullivan minimal models and read generator/differential data
- [x] Certified formality with `is_formal_space` → `FormalityResult`
- [x] Extracted Massey products with `extract_massey_products` → `MasseyProductsResult`
- [x] Computed rational homotopy groups $\pi_n(X) \otimes \mathbb{Q}$

## Next Steps
- **NB 31**: Higher homotopy groups combines these rational results with Adams SS torsion
- **NB 27**: Persistent homology refined can use topological loss to track barcode changes
- **NB 34**: The capstone pipeline uses Sullivan models as a formality pre-check